# Periodic committor: full reproducible pilot

This notebook runs one complete periodic-committor comparison. It is **not** a smoke test. It trains the boundary-adapted exact ansatz, the naïve exact-product ansatz, and the boundary-penalty PINN for 6,000 epochs each; computes 128 and 256-grid finite-difference reference solutions; and saves checkpoints, figures, and a JSON summary to Google Drive.

## Use it

1. Choose **Runtime → Change runtime type → T4 GPU → Save**.
2. Click **Runtime → Run all**.
3. In a fresh Colab runtime, the first cell prompts once for a GitHub token. Paste the repository-read token and press Enter; the input is hidden. This token only lets Colab clone the private repository. It is not used for training and is not stored in the clone.
4. Leave the notebook running. The output folder is printed before training starts. The final cell displays the saved figures.

If Colab disconnects, use the final resume cell with the printed Drive output folder. Do not start a new run directory to resume an old experiment.

In [ ]:
from pathlib import Path
from getpass import getpass
import base64
import shutil
import subprocess

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = 'committor-colab'

if not (REPO / 'pyproject.toml').is_file():
    if REPO.exists():
        shutil.rmtree(REPO)  # only a failed partial clone at this fixed Colab path
    token = getpass('Paste your GitHub token, then press Enter: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    try:
        subprocess.run(
            ['git', '-c', f'http.extraheader=AUTHORIZATION: basic {auth}',
             'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
    finally:
        del token, auth
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
else:
    print('Using the existing clone at', REPO)

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
if not __import__('torch').cuda.is_available():
    raise RuntimeError('This full pilot needs a GPU. Select a T4 GPU under Runtime → Change runtime type, then rerun this notebook.')
subprocess.run(['nvidia-smi'], check=True)
print('Setup passed. The next cell starts the full three-method pilot.')

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess

run_name = 'periodic_committor_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('Full pilot results will be saved in:', RUN_ROOT)

subprocess.run([
    'met-pinn-periodic-comparison',
    'configs/periodic_committor/overnight_comparison.json',
    '--output-dir', RUN_ROOT,
], check=True)

In [ ]:
import json
from IPython.display import Image, display

with open(f'{RUN_ROOT}/summary.json') as handle:
    print(json.dumps(json.load(handle), indent=2))
for filename in ['losses.png', 'committor_boundary_adapted.png', 'committor_naive_product.png', 'committor_penalty.png']:
    display(Image(filename=f'{RUN_ROOT}/{filename}'))

## Resume an interrupted pilot

Rerun the setup cell in a fresh runtime. Then replace the quoted folder below with the `RUN_ROOT` printed by the interrupted run and execute the cell. Each method resumes from its own `last.pt` checkpoint.

In [ ]:
# RUN_ROOT = '/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/periodic_committor_YYYYMMDD_HHMMSS'
# subprocess.run([
#     'met-pinn-periodic-comparison',
#     'configs/periodic_committor/overnight_comparison.json',
#     '--output-dir', RUN_ROOT,
#     '--resume',
# ], check=True)